In [20]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

def locate_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate
    return current

root = locate_root()
data_dir = root / "data"
metadata_dir = data_dir / "metadata"
processed_dir = data_dir / "processed"
reports_dir = root / "reports"
figures_dir = reports_dir / "figures"
tables_dir = reports_dir / "tables"

for directory in [figures_dir, tables_dir]:
    directory.mkdir(parents=True, exist_ok=True)

standardized_df = pd.read_parquet(processed_dir / "flight_rows_standardized.parquet")
feature_manifest = pd.read_csv(metadata_dir / "feature_manifest.csv")
healthy_stats = pd.read_csv(metadata_dir / "healthy_reference_feature_stats.csv")

metadata_columns = ["file_name", "group", "relative_path", "sheet_name", "header_row", "original_row_index", "sequence_index", "time_index", "time_source"]
feature_columns = [col for col in standardized_df.columns if col not in metadata_columns]

if len(feature_columns) == 0:
    raise ValueError("No feature columns were found in flight_rows_standardized.parquet.")

pd.DataFrame(
    {
        "metric": ["rows", "features", "files", "healthy_rows", "defective_rows"],
        "value": [
            len(standardized_df),
            len(feature_columns),
            standardized_df["file_name"].nunique(),
            int((standardized_df["group"] == "healthy").sum()),
            int((standardized_df["group"] == "defective").sum()),
        ],
    }
)

,metric,value
0,rows,312501
1,features,178
2,files,23
3,healthy_rows,142391
4,defective_rows,170110


In [21]:
group_summary = standardized_df.groupby("group", as_index=False).agg(
    files=("file_name", "nunique"),
    rows=("sequence_index", "count"),
    min_time=("time_index", "min"),
    max_time=("time_index", "max"),
)

file_summary = standardized_df.groupby(["group", "file_name"], as_index=False).agg(
    rows=("sequence_index", "count"),
    time_source=("time_source", "first"),
    min_time=("time_index", "min"),
    max_time=("time_index", "max"),
)

file_summary["rows_rank_desc"] = file_summary["rows"].rank(method="dense", ascending=False)

group_summary.to_csv(tables_dir / "eda_group_summary.csv", index=False)
file_summary.to_csv(tables_dir / "eda_file_summary.csv", index=False)

group_summary

,group,files,rows,min_time,max_time
0,defective,14,170110,0.0,15688.0
1,healthy,9,142391,0.0,20461.0


In [22]:
feature_summary_rows = []

healthy_df = standardized_df[standardized_df["group"] == "healthy"]
defective_df = standardized_df[standardized_df["group"] == "defective"]

for feature in tqdm(feature_columns, total=len(feature_columns), desc="Summarizing features"):
    h = healthy_df[feature]
    d = defective_df[feature]
    feature_summary_rows.append(
        {
            "feature": feature,
            "healthy_non_null_ratio": float(h.notna().mean()),
            "defective_non_null_ratio": float(d.notna().mean()),
            "healthy_mean": float(h.mean()) if h.notna().any() else np.nan,
            "defective_mean": float(d.mean()) if d.notna().any() else np.nan,
            "healthy_median": float(h.median()) if h.notna().any() else np.nan,
            "defective_median": float(d.median()) if d.notna().any() else np.nan,
            "healthy_std": float(h.std(ddof=0)) if h.notna().any() else np.nan,
            "defective_std": float(d.std(ddof=0)) if d.notna().any() else np.nan,
            "healthy_abs_gt_3_rate": float((h.abs() > 3).mean()),
            "defective_abs_gt_3_rate": float((d.abs() > 3).mean()),
            "healthy_abs_gt_5_rate": float((h.abs() > 5).mean()),
            "defective_abs_gt_5_rate": float((d.abs() > 5).mean()),
        }
    )

feature_summary = pd.DataFrame(feature_summary_rows)
feature_summary["abs_mean_shift"] = (feature_summary["defective_mean"] - feature_summary["healthy_mean"]).abs()
feature_summary["abs_median_shift"] = (feature_summary["defective_median"] - feature_summary["healthy_median"]).abs()
feature_summary["outlier_rate_gap_3"] = feature_summary["defective_abs_gt_3_rate"] - feature_summary["healthy_abs_gt_3_rate"]
feature_summary["outlier_rate_gap_5"] = feature_summary["defective_abs_gt_5_rate"] - feature_summary["healthy_abs_gt_5_rate"]
feature_summary = feature_summary.sort_values(
    ["abs_mean_shift", "outlier_rate_gap_3", "feature"],
    ascending=[False, False, True],
).reset_index(drop=True)

feature_summary.to_csv(tables_dir / "eda_feature_summary.csv", index=False)
feature_summary.head(20)

Summarizing features: 100%|██████████| 178/178 [00:01<00:00, 108.41it/s]


,feature,healthy_non_null_ratio,defective_non_null_ratio,healthy_mean,defective_mean,healthy_median,defective_median,healthy_std,defective_std,healthy_abs_gt_3_rate,defective_abs_gt_3_rate,healthy_abs_gt_5_rate,defective_abs_gt_5_rate,abs_mean_shift,abs_median_shift,outlier_rate_gap_3,outlier_rate_gap_5
0,rjg,1.0,1.0,62.586441,249.515366,0.0,238.760422,124.637451,98.504051,0.207977,1.000000,0.207977,1.000000,186.928925,238.760422,0.792023,0.792023
1,rjp,1.0,1.0,5.896546,-40.503185,0.0,-41.147358,15.748385,59.790794,0.195939,0.975769,0.188874,0.939192,46.399731,41.147358,0.779829,0.750318
2,rftj,1.0,1.0,-0.007446,5.937075,0.0,0.000000,1.000000,26.601225,0.000801,0.098172,0.000801,0.098172,5.944521,0.000000,0.097371,0.097371
3,fzs2,1.0,1.0,-4.934806,-10.769663,0.0,0.000000,17.405540,33.721313,0.143359,0.166774,0.141996,0.160543,5.834857,0.000000,0.023416,0.018547
4,2fzs2,1.0,1.0,-4.938484,-10.770473,0.0,-0.166662,17.402641,33.713741,0.143253,0.166904,0.141828,0.160379,5.831988,0.166662,0.023650,0.018551
5,2fysy,1.0,1.0,-1.944754,-7.463690,0.0,-2.454636,13.566675,27.799648,0.187919,0.334960,0.014355,0.054012,5.518935,2.454636,0.147041,0.039657
6,fysy,1.0,1.0,-1.814461,-6.794494,0.0,-2.250007,12.450556,25.503401,0.092927,0.185551,0.013779,0.052707,4.980033,2.250007,0.092623,0.038928
7,jhdb,1.0,1.0,-1.676456,-6.143352,0.0,-0.222421,17.289488,34.432755,0.101516,0.160473,0.007255,0.036376,4.466896,0.222421,0.058956,0.029122
8,2jhdb,1.0,1.0,-1.683897,-6.143678,0.0,-0.222421,17.326916,34.435024,0.102183,0.159591,0.007325,0.036471,4.459781,0.222421,0.057407,0.029146
9,jhdc,1.0,1.0,-1.631333,-5.991869,0.0,0.000000,17.256123,34.384571,0.095259,0.153936,0.007255,0.038928,4.360536,0.000000,0.058677,0.031673


In [23]:
healthy_sample = healthy_df.sample(min(len(healthy_df), 30000), random_state=42) if len(healthy_df) else healthy_df.copy()
defective_sample = defective_df.sample(min(len(defective_df), 30000), random_state=42) if len(defective_df) else defective_df.copy()
top_features = feature_summary.head(min(12, len(feature_summary)))["feature"].tolist()

if top_features:
    ncols = 3
    nrows = int(np.ceil(len(top_features) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, feature in tqdm(list(zip(axes, top_features)), total=len(top_features), desc="Plotting feature distributions"):
        h = healthy_sample[feature].dropna().to_numpy()
        d = defective_sample[feature].dropna().to_numpy()
        if len(h):
            ax.hist(h, bins=50, density=True, alpha=0.5, label="healthy")
        if len(d):
            ax.hist(d, bins=50, density=True, alpha=0.5, label="defective")
        ax.set_title(feature)
        ax.legend()
    for ax in axes[len(top_features):]:
        ax.axis("off")
    fig.tight_layout()
    distribution_path = figures_dir / "eda_top_feature_distributions.png"
    fig.savefig(distribution_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
else:
    distribution_path = figures_dir / "eda_top_feature_distributions.png"

str(distribution_path)

Plotting feature distributions: 100%|██████████| 12/12 [00:02<00:00,  5.84it/s]


'/home/hamza/Documents/aero-res/aircraft_fault_localization/reports/figures/eda_top_feature_distributions.png'

In [24]:
correlation_features = feature_summary[feature_summary["healthy_non_null_ratio"] >= 0.7].head(min(25, len(feature_summary)))["feature"].tolist()
healthy_corr_source = healthy_df[correlation_features].fillna(0)
correlation_matrix = healthy_corr_source.corr() if correlation_features else pd.DataFrame()

if not correlation_matrix.empty:
    fig, ax = plt.subplots(figsize=(14, 12))
    image = ax.imshow(correlation_matrix.to_numpy(), aspect="auto")
    ax.set_xticks(range(len(correlation_features)))
    ax.set_xticklabels(correlation_features, rotation=90)
    ax.set_yticks(range(len(correlation_features)))
    ax.set_yticklabels(correlation_features)
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    correlation_figure_path = figures_dir / "eda_healthy_correlation_heatmap.png"
    fig.savefig(correlation_figure_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    pairs = []
    for i in range(len(correlation_features)):
        for j in range(i + 1, len(correlation_features)):
            pairs.append(
                {
                    "feature_1": correlation_features[i],
                    "feature_2": correlation_features[j],
                    "correlation": float(correlation_matrix.iloc[i, j]),
                    "abs_correlation": float(abs(correlation_matrix.iloc[i, j])),
                }
            )
    top_pairs = pd.DataFrame(pairs).sort_values(
        ["abs_correlation", "feature_1", "feature_2"],
        ascending=[False, True, True],
    ).reset_index(drop=True)
else:
    correlation_figure_path = figures_dir / "eda_healthy_correlation_heatmap.png"
    top_pairs = pd.DataFrame(columns=["feature_1", "feature_2", "correlation", "abs_correlation"])

top_pairs.to_csv(tables_dir / "eda_top_correlation_pairs.csv", index=False)
str(correlation_figure_path)

'/home/hamza/Documents/aero-res/aircraft_fault_localization/reports/figures/eda_healthy_correlation_heatmap.png'

In [25]:
file_profile_rows = []

for file_name, file_df in tqdm(standardized_df.groupby("file_name"), total=standardized_df["file_name"].nunique(), desc="Profiling files"):
    block = file_df[feature_columns]
    abs_block = block.abs()
    row_mean_abs = abs_block.mean(axis=1, skipna=True)
    row_max_abs = abs_block.max(axis=1, skipna=True)
    file_profile_rows.append(
        {
            "file_name": file_name,
            "group": file_df["group"].iloc[0],
            "rows": int(len(file_df)),
            "mean_row_mean_abs_z": float(row_mean_abs.mean()) if len(row_mean_abs) else np.nan,
            "p95_row_mean_abs_z": float(row_mean_abs.quantile(0.95)) if len(row_mean_abs) else np.nan,
            "max_row_max_abs_z": float(row_max_abs.max()) if len(row_max_abs) else np.nan,
            "share_rows_any_abs_z_gt_3": float((row_max_abs > 3).mean()) if len(row_max_abs) else np.nan,
            "share_rows_any_abs_z_gt_5": float((row_max_abs > 5).mean()) if len(row_max_abs) else np.nan,
        }
    )

file_profile = pd.DataFrame(file_profile_rows).sort_values(
    ["share_rows_any_abs_z_gt_3", "max_row_max_abs_z"],
    ascending=[False, False],
).reset_index(drop=True)

file_profile.to_csv(tables_dir / "eda_file_profile.csv", index=False)

plot_df = file_profile.copy()
plot_df["file_label"] = plot_df["file_name"]

fig, ax = plt.subplots(figsize=(14, 8))
ax.barh(plot_df["file_label"], plot_df["share_rows_any_abs_z_gt_3"].to_numpy(), alpha=0.8)
ax.invert_yaxis()
ax.set_xlabel("share_rows_any_abs_z_gt_3")
ax.set_ylabel("file_name")
fig.tight_layout()

file_profile_figure_path = figures_dir / "eda_file_outlier_profile.png"
fig.savefig(file_profile_figure_path, dpi=200, bbox_inches="tight")
plt.close(fig)

file_profile.head(20)

Profiling files: 100%|██████████| 23/23 [00:01<00:00, 17.45it/s]


,file_name,group,rows,mean_row_mean_abs_z,p95_row_mean_abs_z,max_row_max_abs_z,share_rows_any_abs_z_gt_3,share_rows_any_abs_z_gt_5
0,Defect16(02).xlsx,defective,14537,4.461068,18.383187,909.252319,1.000000,1.000000
1,Defected 6.xlsx,defective,14537,4.461068,18.383187,909.252319,1.000000,1.000000
2,Defect15(04).xlsx,defective,13123,3.268923,5.063785,603.401001,1.000000,1.000000
3,Defected 4.xlsx,defective,13123,3.268923,5.063785,603.401001,1.000000,1.000000
4,Healthy 8.xlsx,healthy,9152,3.677066,10.584234,556.445129,1.000000,1.000000
5,Defect15(02).xlsx,defective,4902,6.345367,18.604563,545.451477,1.000000,1.000000
6,Defected 2.xlsx,defective,4902,6.345367,18.604563,545.451477,1.000000,1.000000
7,Healthy 9.xlsx,healthy,20462,3.523541,5.041702,498.489288,1.000000,1.000000
8,Defect16(03).xlsx,defective,14683,2.811964,4.212844,389.583923,1.000000,1.000000
9,Defected 7.xlsx,defective,14683,2.811964,4.212844,389.583923,1.000000,1.000000


In [26]:
{
    "eda_group_summary": str(tables_dir / "eda_group_summary.csv"),
    "eda_file_summary": str(tables_dir / "eda_file_summary.csv"),
    "eda_feature_summary": str(tables_dir / "eda_feature_summary.csv"),
    "eda_top_correlation_pairs": str(tables_dir / "eda_top_correlation_pairs.csv"),
    "eda_file_profile": str(tables_dir / "eda_file_profile.csv"),
    "eda_top_feature_distributions": str(figures_dir / "eda_top_feature_distributions.png"),
    "eda_healthy_correlation_heatmap": str(figures_dir / "eda_healthy_correlation_heatmap.png"),
    "eda_file_outlier_profile": str(figures_dir / "eda_file_outlier_profile.png"),
}

{'eda_group_summary': '/home/hamza/Documents/aero-res/aircraft_fault_localization/reports/tables/eda_group_summary.csv',
 'eda_file_summary': '/home/hamza/Documents/aero-res/aircraft_fault_localization/reports/tables/eda_file_summary.csv',
 'eda_feature_summary': '/home/hamza/Documents/aero-res/aircraft_fault_localization/reports/tables/eda_feature_summary.csv',
 'eda_top_correlation_pairs': '/home/hamza/Documents/aero-res/aircraft_fault_localization/reports/tables/eda_top_correlation_pairs.csv',
 'eda_file_profile': '/home/hamza/Documents/aero-res/aircraft_fault_localization/reports/tables/eda_file_profile.csv',
 'eda_top_feature_distributions': '/home/hamza/Documents/aero-res/aircraft_fault_localization/reports/figures/eda_top_feature_distributions.png',
 'eda_healthy_correlation_heatmap': '/home/hamza/Documents/aero-res/aircraft_fault_localization/reports/figures/eda_healthy_correlation_heatmap.png',
 'eda_file_outlier_profile': '/home/hamza/Documents/aero-res/aircraft_fault_localiz